# TESS Exoplanet Pipeline — Step-by-Step Walkthrough

This notebook runs each pipeline stage in order using `TESSAnalysis`.

Install from the **project root** (`tess_exoplanet_pipeline/`, not `notebooks/`):

```bash
cd ../..   # or: cd ~/Desktop/Siya/exoplanet_modelling/tess_exoplanet_pipeline
pip install -e ".[inference,stellar]"
```

The `[inference]` extra installs `exoplanet`, `exoplanet-core`, `pymc`, `celerite2`, and `arviz`.

In [ ]:
# !pip install -e ".[inference,stellar]"


In [ ]:
%matplotlib inline

from tess_pipeline import TESSAnalysis

## Configuration

Create an analysis session. All options map to `PipelineConfig` fields.

In [ ]:
# Option A: download from MAST (target required)
# analysis = TESSAnalysis(
#     "TIC 261136679",
#     inference=True,
#     search_method="tls",
#     sectors=1,
#     cadence=120,
#     chains=2,
#     draws=500,
#     tune=500,
    # chains=1,
    # draws=100,
    # tune=100,
#     plots=True,
#     verbose=True,
# )

# Option B: load local FITS (no target needed — TIC, RA, Dec from FITS headers)
FITS_PATH = "../data/fits/tess2018206045859-s0001-0000000261136679-0120-s_lc.fits"

analysis = TESSAnalysis(
    lightcurve_source="fits",
    lightcurve_fits=[FITS_PATH],
    inference=True,
    search_method="tls",
    plots=True,
    verbose=True,
    chains=1,
    draws=100,
    tune=100
)

## 1. Resolve target

In [ ]:
analysis.resolve_target()
analysis.results.target

## 2. Look up published period (NASA Exoplanet Archive)

In [ ]:
analysis.lookup_archive_period()
analysis.results.period

## 3. Load light curves

In [ ]:
raw = analysis.load_lightcurves()

raw

## 4. Preprocess

In [ ]:
lc = analysis.preprocess()
lc

## 5. Period search (TLS / BLS)

In [ ]:
detection = analysis.search_period()
detection

## 6. Stellar characterization

In [ ]:
analysis.query_gaia()
analysis.characterize_star()
analysis.query_sdss()
analysis.results.stellar

## 7. Bayesian transit fit

In [ ]:
analysis.fit_transit()
analysis.derive_planet_parameters()
analysis.check_convergence()
analysis.results.planet

## 8. Figures and export

In [ ]:
analysis.generate_figures()
analysis.results.summary()
analysis.results.plot_all()

In [ ]:
# Optional: save all outputs to disk
analysis.save("output")

## Alternative: run all stages at once

Equivalent to calling every step above in sequence:

In [ ]:
results = TESSAnalysis(TARGET, inference=True).run()
results.summary()